In [1]:
# -----------------------------
# Install dependencies
# -----------------------------
!pip install transformers accelerate datasets peft bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 12.9 MB/s eta 0:00:00


In [2]:
# -----------------------------
# Imports
# -----------------------------
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from datasets import Dataset
import random

# -----------------------------
# Synthetic dataset (same format as before)
# -----------------------------
instructions = [
    "Customer asks about refund window",
    "Customer wants to cancel an order",
    "Order arrived late",
    "Wrong item received",
    "Product not working",
    "Shipping cost inquiry",
    "Change delivery address",
    "Request for invoice",
    "Ask about warranty",
    "Technical support request"
]

responses = [
    "Our refund window is 30 days from delivery.",
    "You can cancel your order from your account dashboard within 24 hours.",
    "Sorry for the delay. A delivery credit has been applied.",
    "We’ll ship the correct item and provide a return label.",
    "Please try resetting the product. Contact support if the issue persists.",
    "Shipping cost depends on your location and chosen delivery speed.",
    "You can update your delivery address before the order ships.",
    "An invoice will be emailed to you after purchase.",
    "Your product comes with a 12-month warranty.",
    "Our tech support team will contact you shortly."
]

train_data = []
for i in range(300):
    idx = random.randint(0, len(instructions)-1)
    train_data.append({
        "instruction": f"{instructions[idx]} #{i+1}",
        "response": responses[idx]
    })

dataset = Dataset.from_list(train_data)


In [3]:
# -----------------------------
# Model + Tokenizer
# -----------------------------
model_name = "microsoft/phi-2"  # 2.7B instruction-tuned model
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [4]:
# -----------------------------
# BitsAndBytes 4-bit config
# -----------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
# -----------------------------
# Preprocessing
# -----------------------------
def preprocess(example):
    prompt = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['response']}"
    enc = tokenizer(prompt, padding="max_length", truncation=True, max_length=256)
    enc["labels"] = enc["input_ids"].copy()
    return enc

tokenized_dataset = dataset.map(preprocess)
tokenized_dataset.set_format(type="torch", columns=["input_ids","attention_mask","labels"])

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [6]:
# -----------------------------
# LoRA config
# -----------------------------
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

trainable params: 2,621,440 || all params: 2,782,305,280 || trainable%: 0.0942


In [7]:
# -----------------------------
# TrainingArguments
# -----------------------------
training_args = TrainingArguments(
    output_dir="./phi2-qlora-lora",
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    logging_steps=5,
    save_strategy="no",
    report_to="none",  # disables W&B
    fp16=True,
)

In [8]:
# -----------------------------
# Trainer
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

Step,Training Loss
5,6.895200
10,5.689400
15,2.941800
20,0.700100
25,0.463400
30,0.400700
35,0.350900
40,0.315400
45,0.253000
50,0.213200


TrainOutput(global_step=225, training_loss=0.44813723974757724, metrics={'train_runtime': 242.8812, 'train_samples_per_second': 3.706, 'train_steps_per_second': 0.926, 'total_flos': 3665064886272000.0, 'train_loss': 0.44813723974757724, 'epoch': 3.0})

In [9]:
# -----------------------------
# Generate helper
# -----------------------------
def generate(prompt):
    text = f"### Instruction:\n{prompt}\n\n### Response:\n"
    inp = tokenizer(text, return_tensors="pt").to(device)
    out = model.generate(
        **inp,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [10]:
# -----------------------------
# Test prompts
# -----------------------------
for p in [
    "Customer asks about refund window",
    "Order arrived late",
    "Wrong item received",
    "How do I request an invoice?",
    "Product is not working"
]:
    print("----")
    print(generate(p))

----
### Instruction:
Customer asks about refund window

### Response:
Our refund window is 30 days after delivery.
----
### Instruction:
Order arrived late

### Response:
Sorry for the delay. A delivery credit has been applied.
----
### Instruction:
Wrong item received

### Response:
We’ll ship the correct item and provide a return label.
----
### Instruction:
How do I request an invoice?

### Response:
You can request an invoice from your invoice email.
----
### Instruction:
Product is not working

### Response:
Please try resetting the product. Contact support if the issue persists.
